# <font color="steelblue">Patrones de demanda</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a>

No olvides hacer una copia si deseas utilizarlo.

> **Guión de trabajo para el estudiante.** Este cuaderno es el **enunciado** de un proyecto de **aprendizaje no supervisado** que debéis **desarrollar vosotros**: objetivos, fases, tareas obligatorias, andamiaje mínimo de código (marcas `# TODO`) y criterios de evaluación. **No es una solución.**

In [ ]:
# @title Cargar configuración del cuaderno
!pip install gdown
!pip install --upgrade kagglehub
!pip install lightgbm xgboost
!pip install ucimlrepo

# Análisis numérico
import numpy as np
import pandas as pd
import math, random
import warnings
warnings.filterwarnings('ignore')

# Gráficos
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set_theme(style='whitegrid')
%config InlineBackend.figure_format = 'retina'

import os, zipfile, gdown, kagglehub

# Funciones del curso
from urllib.request import urlretrieve
for fichero in ['preprocesar.py', 'evaluar_clasificadores.py', 'pca_funciones.py', 'auto_ML.py']:
    urlretrieve(f'https://raw.githubusercontent.com/ia4legos/MachineLearning/refs/heads/main/{fichero}', fichero)
from preprocesar import *
from evaluar_clasificadores import *
from pca_funciones import *
from auto_ML import *

from sklearn.decomposition import PCA, KernelPCA, IncrementalPCA, FastICA
from sklearn.cluster import KMeans, MiniBatchKMeans, BisectingKMeans
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import AgglomerativeClustering, FeatureAgglomeration
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS, Isomap, LocallyLinearEmbedding, TSNE, SpectralEmbedding

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (silhouette_score, calinski_harabasz_score, davies_bouldin_score,
                             accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             adjusted_rand_score)

RNG = 42

## Cargar funciones accesorias

############# Funciones MDS ##############
def stress1(D_orig, embedding):
    """Stress-1 de Kruskal entre las distancias originales y las del embedding.
    Portable entre versiones de scikit-learn (no depende del atributo stress_)."""
    D_proj = pairwise_distances(embedding)
    iu = np.triu_indices_from(D_orig, k=1)          # pares (i<j), sin diagonal
    num = np.sum((D_orig[iu] - D_proj[iu])**2)
    den = np.sum(D_proj[iu]**2)
    return np.sqrt(num / den)

def ajustar_mds(D, n_components=2, metrico=True, n_init=4, random_state=RNG):
    """Ajusta un MDS (métrico o no) sobre una matriz de distancias precalculada
    y devuelve el embedding. Usa 'metric'/'dissimilarity' por compatibilidad amplia."""
    modelo = MDS(n_components=n_components, dissimilarity='precomputed',
                 metric=metrico, n_init=n_init, random_state=random_state)
    return modelo.fit_transform(D)

def curva_stress(D, metrico=True, max_dim=10, n_init=4):
    """Calcula el Stress-1 para soluciones de 1 a max_dim dimensiones."""
    dims = range(1, max_dim + 1)
    valores = [stress1(D, ajustar_mds(D, d, metrico, n_init)) for d in dims]
    return list(dims), valores

def grafico_codo(dims, valores, titulo=''):
    """Curva de Stress-1 vs nº de dimensiones, con las bandas de calidad de Kruskal."""
    plt.figure(figsize=(8, 5))
    plt.plot(dims, valores, 'o-')
    for y, txt, col in [(0.05, 'bueno', 'green'), (0.10, 'aceptable', 'orange'), (0.20, 'pobre', 'red')]:
        plt.axhline(y, ls='--', lw=1, color=col, alpha=.6)
        plt.text(dims[-1], y, f'  {txt}', color=col, va='center')
    plt.xticks(dims); plt.xlabel('Nº de dimensiones'); plt.ylabel('Stress-1')
    plt.title(titulo); plt.show()

def grafico_distancias(D_orig, embedding, titulo='Distancias proyectadas vs originales'):
    """Gráfico distancia-distancia: nube sobre la diagonal => buen ajuste."""
    D_proj = pairwise_distances(embedding)
    m = np.max(D_orig)
    plt.figure(figsize=(8, 6))
    plt.scatter(D_orig, D_proj, s=8, alpha=.3, label='pares de puntos')
    plt.plot([0, m], [0, m], 'k-', lw=2, label='ajuste perfecto (y = x)')
    plt.xlabel('Distancia original'); plt.ylabel('Distancia proyectada')
    plt.title(titulo); plt.legend(); plt.show()

def nearest_neighbors(X, k):
    """Matriz (n, k) con los índices de los k vecinos más próximos de cada punto
    (distancia euclídea), excluyendo el propio punto. Devuelve enteros."""
    D = pairwise_distances(X)
    # ordena cada fila por distancia; la columna 0 es el propio punto (dist. 0)
    return np.argsort(D, axis=1)[:, 1:k+1]

def dibujar_grafo(X, knn, ax=None, **kw):
    """Dibuja las aristas punto–vecino sobre un eje 2D."""
    ax = ax or plt.gca()
    for i, vecinos in enumerate(knn):
        for j in vecinos:
            ax.plot(X[[i, j], 0], X[[i, j], 1], c='gray', lw=0.6, zorder=1, **kw)

########### Cluster ##############
def plot_dendrogram(model, **kwargs):
    """Dibuja el dendrograma a partir de un AgglomerativeClustering ya ajustado
    con compute_distances=True (o distance_threshold). Acepta los mismos kwargs
    que scipy.cluster.hierarchy.dendrogram (truncate_mode, p, color_threshold, ax...)."""
    counts = np.zeros(model.children_.shape[0])     # nº de muestras bajo cada fusión
    n = len(model.labels_)
    for i, merge in enumerate(model.children_):
        c = 0
        for child in merge:
            c += 1 if child < n else counts[child - n]   # hoja (1) o sub-cluster (su recuento)
        counts[i] = c
    Z = np.column_stack([model.children_, model.distances_, counts]).astype(float)
    return dendrogram(Z, **kwargs)

def plot_scree(X, metric, linkage_m, kmin, kmax, **kwargs):
    """Índice de silueta para soluciones jerárquicas de kmin a kmax grupos."""
    ks = range(kmin, kmax)
    sil = [silhouette_score(X, AgglomerativeClustering(n_clusters=k, metric=metric,
                                                       linkage=linkage_m).fit_predict(X)) for k in ks]
    plt.plot(list(ks), sil, marker='o', **kwargs)
    plt.xlabel('número de grupos'); plt.ylabel('silueta media'); plt.xticks(list(ks))

from matplotlib.patches import Ellipse
def draw_ellipse(position, cov2x2, ax=None, **kwargs):
    """Dibuja la elipse (a 1, 2 y 3 sigmas) dada una covarianza 2x2."""
    ax = ax or plt.gca()
    U, s, Vt = np.linalg.svd(cov2x2)
    angle = np.degrees(np.arctan2(U[1, 0], U[0, 0]))
    width, height = 2 * np.sqrt(s)
    for nsig in range(1, 4):
        ax.add_patch(Ellipse(position, nsig * width, nsig * height,
                             angle=angle, **kwargs))   # 'angle' es keyword en matplotlib actual

def _cov_2x2(gmm, k):
    """Reconstruye la covarianza 2x2 de la componente k según covariance_type."""
    ct = gmm.covariance_type
    if ct == 'full':      return gmm.covariances_[k]
    if ct == 'tied':      return gmm.covariances_
    if ct == 'diag':      return np.diag(gmm.covariances_[k])
    return np.eye(2) * gmm.covariances_[k]            # 'spherical'

def plot_gmm(gmm, X, label=True, ax=None):
    """Ajusta el GMM y dibuja puntos coloreados por componente + sus elipses
    (válido para los cuatro covariance_type)."""
    ax = ax or plt.gca()
    labels = gmm.fit(X).predict(X)
    ax.scatter(X[:, 0], X[:, 1], c=labels if label else None,
               s=20, cmap='viridis', zorder=2)
    ax.set_aspect('equal')
    w_factor = 0.3 / gmm.weights_.max()
    for k, (pos, w) in enumerate(zip(gmm.means_, gmm.weights_)):
        draw_ellipse(pos, _cov_2x2(gmm, k), ax=ax, alpha=w * w_factor, color='steelblue')

def draw_ellipsoid(ax, mean, cov, nsig=2, **kw):
    # dibuja el elipsoide a 'nsig' sigmas de una gaussiana 3D (media + cov 3x3)
    vals, vecs = np.linalg.eigh(cov)                 # autovalores/autovectores
    u = np.linspace(0, 2*np.pi, 24); v = np.linspace(0, np.pi, 16)
    esfera = np.array([np.outer(np.cos(u), np.sin(v)),
                       np.outer(np.sin(u), np.sin(v)),
                       np.outer(np.ones_like(u), np.cos(v))])
    r = nsig * np.sqrt(vals)                          # longitudes de los semiejes
    pts = np.einsum('ij,jkl->ikl', vecs, esfera * r[:, None, None])
    ax.plot_surface(pts[0]+mean[0], pts[1]+mean[1], pts[2]+mean[2], **kw)

## <font color="steelblue">Objetivos del proyecto</font>

Cada producto es una **serie temporal**: sus ventas a lo largo de 52 semanas dibujan un **patrón de demanda** (plano, estacional, creciente, decreciente, con picos…). El objetivo es **agrupar los productos por la forma de su patrón** (*time series clustering*), no por cuánto venden. El reto central, propio de este proyecto, es trabajar con **series temporales**:

* **Normalizar cada serie** para capturar la **forma** y no el **nivel** (si no, agruparás por volumen de ventas).
* Elegir la **representación** (serie normalizada vs **características** extraídas) y la **distancia** (euclídea vs **DTW**, *Dynamic Time Warping*).
* **Agrupar** con métodos específicos de series (**TimeSeriesKMeans**, **k-Shape**) y compararlos.
* **Caracterizar** y **nombrar** los patrones prototípicos y traducirlos en **gestión de inventario/promociones**.
* Construir un **widget interactivo en Colab** que explore los patrones.

## <font color="steelblue">El conjunto de datos</font>

**811 productos** × **52 semanas** (UCI *Sales Transactions Weekly*; Tan & Lau, 2014). **Sin valores perdidos**; todo **entero ≥ 0**.

* **`Product_Code`** — código del producto. **Eliminar antes de agrupar.**
* **`W0`, `W1`, …, `W51`** — unidades compradas en cada semana del año. En conjunto, la **serie de demanda** del producto.

> **Idea clave:** dos productos pueden tener el **mismo patrón** (p. ej. pico en Navidad) aunque uno venda 10× más que el otro. Como queremos agrupar por **forma**, casi siempre habrá que **normalizar cada serie** antes de medir distancias.
> *(La versión UCI original incluye columnas `Normalized`/`MIN`/`MAX`; en la del curso solo están `W0..W51`.)*

## <font color="steelblue">Reglas del juego</font>

1. **Quita `Product_Code`** y trabaja con la matriz de 52 semanas.
2. **Normaliza por serie** (cada producto respecto a su propia media/desv o min–max) para agrupar por **forma**, no por **nivel**. Justifica la elección y compara con no normalizar.
3. **Combina enfoques:** representa las series de **dos formas** (serie normalizada cruda vs **características** extraídas) y compara los grupos resultantes.
4. **Usa al menos un método específico de series** (DTW / **k-Shape**) además de uno clásico, y compáralos.
5. **Elige el número de grupos con criterios** (silueta, codo, dendrograma), no a ojo.
6. **Caracteriza los patrones:** dibuja la **forma prototípica** de cada grupo y **nómbralo**; sin esto el proyecto no vale.
7. **Reproducibilidad:** fija `random_state` y comenta la **estabilidad**.

# <font color="steelblue">Fase 0 — Preparación del entorno y carga de datos</font>

In [ ]:
url = "https://raw.githubusercontent.com/ia4legos/MachineLearning/main/data/Sales_Transactions_Weekly.csv"
sales = pd.read_csv(url)
print(f"Dimensiones: {sales.shape[0]:,} productos × {sales.shape[1]} columnas")
sales.head()

# <font color="steelblue">Fase 1 — Comprensión y EDA temporal</font>

**Tareas a desarrollar**
1. **Estructura.** Confirmad las 52 columnas semanales `W0..W51` + `Product_Code`.
2. **Visualización de series.** Dibujad las series de **una muestra** de productos (varias en un mismo gráfico). Observad la diversidad de patrones (planos, con picos, estacionales…).
3. **Nivel vs forma (clave).** Comparad el **volumen total** por producto (suma/medias): veréis que los **niveles** varían enormemente. ¿Qué pasaría si agruparais sin normalizar?
4. **Estadísticos temporales.** Semana media de máximo, % de semanas con 0 ventas, variabilidad… para intuir tipos de patrón.
5. **Conclusión:** 3–4 hallazgos.

> **A responder:** ¿por qué, si no normalizas, el *clustering* separará "productos que venden mucho" de "productos que venden poco" en lugar de **patrones**?

# <font color="steelblue">Fase 2 — Normalización por serie (forma vs nivel)</font>

**Tareas a desarrollar**
1. Construid la matriz `X` de series (`W0..W51`, sin `Product_Code`).
2. **Normalizad cada serie por separado** para que importe la **forma**:
   * **z-normalización** por fila (resta su media, divide por su desv.), o
   * **min–max** por fila, o
   * la `TimeSeriesScalerMeanVariance` de **tslearn**.
3. **Comparad** el efecto: dibujad la misma muestra de series **antes** y **después** de normalizar.
4. (Opcional) Suavizado ligero (media móvil) para reducir ruido semanal.

> **A responder:** ¿qué información **pierdes** al normalizar (el nivel) y por qué aquí **interesa** perderla?

# <font color="steelblue">Fase 3 — Representación y reducción de dimensión</font>

**Tareas a desarrollar**
1. **Representación A — serie normalizada cruda** (52 valores por producto).
2. **Representación B — características extraídas:** resumid cada serie con variables interpretables (p. ej. **pendiente/tendencia**, **fuerza estacional**, **nº de picos**, **autocorrelación lag-1**, **coef. de variación**, semana del máximo, % de ceros). Esto convierte el problema en un *clustering* tabular clásico.
3. **Visualización.** Aplicad **PCA** y/o **MDS**/**t-SNE** sobre la representación A para ver si hay estructura.
4. Comentad qué representación parece capturar mejor los patrones.

> **Combinar modelos:** en la Fase 4 agruparéis con **ambas** representaciones y compararéis.

# <font color="steelblue">Fase 4 — Clustering de series y combinación de modelos</font>

**Tareas a desarrollar**
1. **Método específico de series:** **`TimeSeriesKMeans`** de tslearn con distancia **euclídea** y con **DTW** (*Dynamic Time Warping*), y/o **`KShape`** (basado en correlación de formas).
2. **Método clásico sobre características:** K-Means / jerárquico sobre la **representación B**.
3. **Combina/compara:** ¿agrupan igual la serie normalizada (euclídeo/DTW/k-Shape) y las características? Mide la **concordancia** (Adjusted Rand Index).
4. Dibuja, para una solución, los **centroides** (formas medias) de cada grupo.

> **A responder:** ¿qué aporta **DTW** frente a la distancia euclídea cuando dos series tienen el mismo patrón **desfasado** en el tiempo?

# <font color="steelblue">Fase 5 — Validación y elección del número de grupos</font>

**Tareas a desarrollar**
1. **Número de grupos:** **codo** (inercia) y **silueta** (con distancia euclídea y, si puedes, **DTW**) variando `k`; **dendrograma** para el jerárquico.
2. **Índices internos** y comparación entre métodos/representaciones.
3. **Estabilidad** ante semillas/submuestras.
4. **Decisión justificada** del método, la representación y el número de patrones.

> El mejor `k` equilibra **calidad** (índices) e **interpretabilidad** (que los patrones sean distinguibles y con sentido de negocio).

# <font color="steelblue">Fase 6 — Caracterización de los patrones</font>

**Tareas a desarrollar**
1. **Forma prototípica.** Para cada grupo, dibuja la **serie media** (centroide) y una **banda** (± desv.): ese es el **patrón** del grupo.
2. **Etiqueta cada patrón:** plano/estable, **estacional**, creciente, decreciente, **con pico puntual**, intermitente (muchos ceros)…
3. **Tamaño y ejemplos.** Nº de productos por patrón y algunos productos representativos.
4. **Lectura de negocio.** ¿Qué implica cada patrón para **inventario**, **previsión** y **promociones** (p. ej. los estacionales requieren *stock* anticipado; los intermitentes, otra política)?

> Es la fase con **más peso**: el valor está en **nombrar y explicar** los patrones.

# <font color="steelblue">Fase 7 — Aplicación interactiva (widget en Colab)</font>

Aplicación **dentro del cuaderno** con **`ipywidgets`** (no despliegue). Implementa **al menos una**:

1. **Explorador de patrones:** un desplegable de grupo que muestre su **forma prototípica**, su tamaño y varios productos de ejemplo.
2. **Buscador por producto:** elige un `Product_Code` y muestra **su** serie, el **patrón** al que pertenece y la **recomendación** de gestión asociada a ese patrón.
3. **(Opcional)** Dispersión PCA/MDS coloreada por patrón con `plotly`.

> El widget solo **usa** el modelo ya ajustado; no reentrena en cada interacción.

# <font color="steelblue">Entregables y formato</font>

* **Cuaderno** ejecutable de principio a fin, con código y **celdas de texto** que justifiquen cada decisión (normalización, distancia, nº de patrones, interpretación).
* **Memoria breve:** EDA temporal, normalización, representaciones y reducción, comparación de métodos de *clustering* de series, validación, **caracterización y nombres de los patrones**, y la aplicación.
* **Widget(s)** funcionando dentro del cuaderno.
* **Reproducibilidad:** `random_state` fijado y comentario de estabilidad.

## <font color="steelblue">Rúbrica de evaluación (100 pts)</font>

| Fase | Qué se valora | Puntos |
|---|---|---:|
| 1. EDA temporal | Visualización de series, **nivel vs forma**, estadísticos | 12 |
| 2. Normalización por serie | **Forma vs nivel**, comparación antes/después, justificación | 16 |
| 3. Representación y reducción | Serie normalizada vs **características**, PCA/MDS | 14 |
| 4. Clustering de series | Euclídeo + **DTW/k-Shape** + por características, comparación | 18 |
| 5. Validación y nº de grupos | Codo/silueta(DTW)/dendrograma, índices, **estabilidad** | 12 |
| 6. Caracterización de patrones | **Formas prototipo**, nombres, lectura de inventario | 18 |
| 7. Aplicación (widget) | `ipywidgets` funcional (explorador / buscador) | 10 |
| — Extra | *DTW barycenter*, k-Shape, características con `tsfresh`, euclídeo vs DTW, mapa `plotly` | +10 |

> **Penalizaciones:** **agrupar sin normalizar** (acabar segmentando por volumen), elegir el nº de grupos **a ojo**, no dibujar/nombrar las **formas prototipo**, o ignorar que los datos son **series temporales**.

# <font color="steelblue">Pistas y errores típicos</font>

* **Normaliza por serie.** Sin ello agruparás "best-sellers vs nicho", no patrones de demanda.
* **DTW para desfases.** Si dos series tienen el mismo patrón desplazado unas semanas, la euclídea las separa y **DTW** las junta.
* **Dos representaciones.** La serie cruda (con DTW/k-Shape) capta la forma; las **características** (tendencia, estacionalidad, picos) dan interpretabilidad. Compáralas.
* **Caracterizar = dibujar la forma.** El centroide de cada grupo **es** el patrón: nómbralo (estacional, plano, pico…).
* **Negocio:** cada patrón implica una política de **inventario/promoción** distinta.
* **Widget, no despliegue:** `ipywidgets` dentro de Colab.

# <font color="steelblue">Referencias</font>

* Tan, S. C. & Lau, J. P. S. (2014). *Time series clustering: A superior alternative for market basket analysis*. DaEng-2013, Springer.
* *Sales Transactions Dataset Weekly*. UCI ML Repository (id 396).
* Tavenard, R. et al. *tslearn: A Machine Learning Toolkit for Time Series Data*.
* Cuadernos del curso: *Componentes principales y variantes*, *Escalas multidimensionales (MDS)*, *Clustering (K-Means, jerárquico, DBSCAN, mixturas gaussianas)*.